# Fine-tuning RoBERTa with LoRA

**Question (Further Work, conjecture 2 in `fine-tuning-results.md`):** is LoRA about as good as fine-tuning every weight?

Qwen3-Embedding-4B can't answer that yet. LoRA (78.68) tied the frozen model, which fits two stories: LoRA fell short, or fine-tuning doesn't help Qwen at all. RoBERTa-base is a cleaner test, because fine-tuning every weight clearly helps it: frozen + MLP 68.33, fully fine-tuned 72.69 (mean of all 4 runs) and 73.13 (mean of the 3 `sweep.py` seeds).

This is `Fine-tuned Roberta.ipynb` with one change: RoBERTa's 124M weights are frozen, and LoRA's small matrices train instead, along with the head. `LoRA Qwen3-Embedding-4B.ipynb` explains LoRA and the `LoRALinear` class, reused here unchanged, so the comments explain only what's new.

**Run on 2026-11-02** by `../sweeps/sweep_logs/run_lora_roberta.py`, which executes this notebook with the settings cell replaced and saves the printed output in `../sweeps/sweep_logs/`. So the cells here have no outputs. The cost below is the estimate made before running; "Result" at the end has the measured numbers.

**Predict before running:** if LoRA's $B$ matrices stayed at 0, this would be frozen RoBERTa with a trained head: about 68. Fine-tuning every weight got about 73, and the papers in `fine-tuning-results.md` found that LoRA on all layers ties full fine-tuning on GLUE. Where will it land?

## What changes from the full fine-tune

1. **RoBERTa is frozen.** The optimizer gets only LoRA's $A$ and $B$ matrices and the head: 3.3M numbers, 2.6% of the model. The head alone is 0.6M of them.
2. **LoRA on all 6 matrices of all 12 layers:** query, key, value and the attention output, plus the feed-forward's two (768 → 3,072 → 768). Rank $r = 16$, $\alpha = 32$ and dropout 0.05, as in the Qwen notebook. The LoRA paper adapted only query and value, but QLoRA (Section 4) and Schulman et al. ("LoRA Without Regret") found that LoRA needs every layer, the feed-forward ones especially, to match full fine-tuning.
3. **Learning rate 2e-4 instead of 2e-5.** Schulman et al. found the best LoRA learning rate is consistently about 10× full fine-tuning's (the Qwen notebooks used the same ratio: 1e-4 vs. 1e-5). The LoRA paper used 4e-4 to 5e-4 for RoBERTa-base, but with $\alpha / r = 1$. The change to the weights per step scales with $\alpha / r$ times the learning rate, so with our $\alpha / r = 2$, 2e-4 here is about 4e-4 there.
4. **The head learns at the same 2e-4,** as in the Qwen notebook and in Hugging Face's `peft`.

Everything else is the full fine-tune's: the data and thread-grouped dev split, length-grouped batches of 16, Hugging Face's head, the RoBERTa recipe (10 epochs, weight decay 0.1, 6% warmup, β₂ 0.98, ε 1e-6, the epoch with the best dev accuracy), and 32-bit weights with bfloat16 matrix multiplies.

**Expected cost (not measured):**
- **Memory: about 5–6 GB,** against the full fine-tune's 5.2 GB. The frozen weights need no gradients and no AdamW averages (1.5 GB less), but every LoRA layer keeps its own copy of its input for the backward pass (up to about 2 GB more, on the longest batches).
- **Time: probably 35–45 s per epoch,** against 46 s. The backward pass skips the frozen weights' gradients, about a third of the computation, but the 72 small LoRA paths add some back. That's 6–8 minutes per run.

## How to run it: the same tuning effort as the full fine-tune

The full fine-tune's learning rate was picked from 1e-5, 2e-5 and 3e-5 on dev (seed 423), and then 3 seeds were scored on test (`fine-tuning-results.md`). A fair comparison gives LoRA the same:
1. **Choose the learning rate on dev.** Run the notebook 3 times with `SEED = 423` and `LR` = 1e-4, 2e-4 and 5e-4, keeping `SCORE_TEST = False`. Each run appends its dev scores to `../sweeps/sweep.jsonl` and saves its dev and test probabilities in `../sweeps/probs/`, without scoring test.
2. `~/venvs/cs423/bin/python ../sweeps/summarize.py` lists every run's dev accuracy by epoch. Take the learning rate with the highest best-epoch dev accuracy (on a tie, 2e-4). Decide before step 3.
3. Run the chosen learning rate again with `SEED = 1` and `SEED = 2`, the other seeds `sweep.py` used.
4. `~/venvs/cs423/bin/python ../sweeps/summarize.py --final lora_lr2e-4 --select acc` (with the chosen rate's name) scores the 3 seeds and their ensemble on test. The full fine-tune, with `--final r_lr2e-5`: 72.84, 73.11 and 73.45 (mean 73.13), ensemble 73.93.

**Decide now what "about as good" means.** Proposed: LoRA's 3-seed mean within 1 point of 73.13 is a tie; more than 2 points below means LoRA is worse; anything in between needs more seeds.

That's 5 runs, about 35 minutes in all.

In [ ]:
## Settings for this run (see "How to run it" above)
LR = 2e-4           ## 1e-4, 2e-4 and 5e-4 while choosing the learning rate on dev
SEED = 423          ## 423 while choosing the learning rate, then 1 and 2
SCORE_TEST = False  ## True prints the test accuracy at the end; summarize.py --final scores all the seeds at once
NAME = f"lora_lr{LR:.0e}".replace("e-0", "e-")   ## "lora_lr2e-4": this run's name in sweep.jsonl and probs/
print(NAME, "seed", SEED)

## Data and dev split

As in the full fine-tune, whose comments explain the Python: headers, footers and quotes removed, empty posts dropped, and the same 1,109 thread-grouped dev posts. The split keeps `random_state=423` whatever `SEED` is, so every run is scored on the same dev posts.

In [ ]:
import re
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import StratifiedGroupKFold

newsgroups_train = fetch_20newsgroups(
    subset='train', remove=('headers', 'footers', 'quotes'))

newsgroups_test = fetch_20newsgroups(
    subset='test', remove=('headers', 'footers', 'quotes'))

# Remove empty documents
for s, name in zip([newsgroups_train, newsgroups_test], ["train", "test"]):
  empty_indices = {i for i, doc in enumerate(s.data) if len(doc) == 0}
  orig_len = len(s.data)
  for k in ['data', 'filenames', 'target']:
    s[k] = [s[k][i] for i in range(orig_len) if i not in empty_indices]
  print(f"Removed {len(empty_indices)} empty documents from the {name} set. Before: {orig_len}. After: {len(s.data)}.")

y_train = np.array(newsgroups_train.target)      ## (11096,)
y_test = np.array(newsgroups_test.target)        ## (7370,)
n_classes = len(newsgroups_train.target_names)   ## 20

## Dev set: 10% of the threads (a thread = the subject line without "Re:"), the same split as the other notebooks
with_headers = fetch_20newsgroups(subset='train')
subject_of = {f: re.search(r"^Subject:(.*)$", doc, re.M).group(1)
              for f, doc in zip(with_headers.filenames, with_headers.data)}
threads = [re.sub(r"^(re:\s*)+", "", subject_of[f].strip(), flags=re.IGNORECASE).lower()
           for f in newsgroups_train.filenames]
thread_ids = np.unique(threads, return_inverse=True)[1]
splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=423)   ## 423, not SEED: the same dev set every run
train_idx, dev_idx = next(splitter.split(newsgroups_train.data, y_train, groups=thread_ids))
X_tr, y_tr = [newsgroups_train.data[i] for i in train_idx], y_train[train_idx]   ## 9,987 posts
X_dev, y_dev = [newsgroups_train.data[i] for i in dev_idx], y_train[dev_idx]     ## 1,109 posts
print(f"Training part: {len(X_tr):,} posts, dev: {len(X_dev):,} posts")

## Tokenizing and batching

As in the full fine-tune: tokenized in the DataLoader, length-grouped training batches of 16, evaluation batches of 64. `torch.manual_seed(SEED)` replaces the fixed 423, so `SEED` changes the batch order, the starting values of the head and of the $A$'s, and the dropout.

**Shapes:**
- **N** = posts in a batch (16 in training, 64 in evaluation). The full fine-tune called it B, but here $B$ is one of LoRA's matrices.
- **T** = tokens in the batch's longest post, at most 512
- **768** = the size of RoBERTa's vectors; **20** = classes

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 512
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

## A list of N (text, label) pairs -> input_ids (N, T), attention_mask (N, T), labels (N,)
def collate(batch):
    texts, labels = zip(*batch)
    enc = tokenizer(list(texts), padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"], torch.tensor(labels)

train_lengths = torch.tensor([len(ids) for ids in tokenizer(X_tr, truncation=True, max_length=MAX_LENGTH)["input_ids"]])

## Batches of posts of similar length, in a new random order every epoch (the full fine-tune explains how)
class LengthGroupedBatches:
    def __init__(self, lengths, batch_size):
        self.lengths, self.batch_size = lengths, batch_size

    def __len__(self):
        return -(-len(self.lengths) // self.batch_size)

    def __iter__(self):
        order = torch.randperm(len(self.lengths))
        batches = []
        for mega in order.split(50 * self.batch_size):
            mega = mega[self.lengths[mega].argsort(descending=True)]
            batches += mega.split(self.batch_size)
        for i in torch.randperm(len(batches)):
            yield batches[i].tolist()

train_loader = DataLoader(list(zip(X_tr, y_tr)), batch_sampler=LengthGroupedBatches(train_lengths, 16),
                          collate_fn=collate)                                                          ## 625 batches
dev_loader = DataLoader(list(zip(X_dev, y_dev)), batch_size=64, collate_fn=collate)                    ## 18 batches
test_loader = DataLoader(list(zip(newsgroups_test.data, y_test)), batch_size=64, collate_fn=collate)   ## 116 batches

input_ids, attention_mask, labels = next(iter(train_loader))   ## one training batch, for the check in the model cell
print("token ids:", tuple(input_ids.shape), " attention mask:", tuple(attention_mask.shape), " labels:", tuple(labels.shape))

## The model: frozen RoBERTa + LoRA + Hugging Face's head

- **The full fine-tune's `RobertaClassifier`, unchanged:** `roberta-base` without its pooler, then dropout → Linear(768, 768) → tanh → dropout → Linear(768, 20) on `<s>`.
- **RoBERTa is frozen** with `model.roberta.requires_grad_(False)`. The head isn't inside `model.roberta`, so all of it still trains. The embedding table, the LayerNorms and the biases stay frozen too, as in standard LoRA and the Qwen notebook.
- **6 matrices per layer get a `LoRALinear`,** 72 in all. Their names in Hugging Face's RoBERTa, and their shapes:

| in each of the 12 layers | $W$ | $A$ | $B$ |
|---|---|---|---|
| `attention.self.query`, `.key`, `.value`, `attention.output.dense` | 768 × 768 | 16 × 768 | 768 × 16 |
| `intermediate.dense` (feed-forward in) | 3,072 × 768 | 16 × 768 | 3,072 × 16 |
| `output.dense` (feed-forward out) | 768 × 3,072 | 16 × 3,072 | 768 × 16 |

- **The frozen weights stay 32-bit,** as in the full fine-tune, so the arithmetic matches it: inside `torch.autocast`, every matrix multiply runs in bfloat16 either way. (The Qwen notebook stored its frozen weights in bfloat16 to fit in memory. RoBERTa-base is small enough not to need that.)
- **Sanity check:** $B$ starts at 0, so right after wrapping, RoBERTa must compute exactly what it did before. The cell checks this on one batch.

In [ ]:
from transformers import AutoModel

R, ALPHA, LORA_DROPOUT = 16, 32, 0.05

## The Qwen notebook's LoRALinear, unchanged: a frozen nn.Linear plus (alpha / r) B A x
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout):   ## base: the pre-trained layer, whose weight W is (d_out, d_in)
        super().__init__()
        self.base = base
        self.A = nn.Linear(base.in_features, r, bias=False, device=base.weight.device)   ## A: (r, d_in), random
        self.B = nn.Linear(r, base.out_features, bias=False, device=base.weight.device)  ## B: (d_out, r)
        nn.init.zeros_(self.B.weight)    ## B = 0, so B A = 0: the layer starts out computing exactly W x
        self.dropout = nn.Dropout(dropout)
        self.scale = alpha / r           ## 32 / 16 = 2

    def forward(self, x):                ## x: (N, T, d_in)
        return self.base(x) + self.B(self.A(self.dropout(x))) * self.scale   ## (N, T, d_out)

## The full fine-tune's model, unchanged
class RobertaClassifier(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.roberta = AutoModel.from_pretrained("roberta-base", add_pooling_layer=False)
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Sequential(
            nn.Linear(768, 768), nn.Tanh(),     ## new and random: (N, 768) -> (N, 768)
            nn.Dropout(0.1),
            nn.Linear(768, n_classes))          ## new and random: (N, 768) -> (N, 20)

    def forward(self, input_ids, attention_mask):    ## both (N, T)
        hidden = self.roberta(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state   ## (N, T, 768)
        return self.head(self.dropout(hidden[:, 0]))  ## <s>: (N, 768) -> (N, 20)

model = RobertaClassifier(n_classes).to(device)

## The example batch's <s> vectors before LoRA is added, in eval mode (dropout off), for the check below
ids, mask = input_ids.to(device), attention_mask.to(device)
model.eval()
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    s_before = model.roberta(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0]   ## (N, 768)

model.roberta.requires_grad_(False)   ## freeze all of RoBERTa's weights; the head isn't in model.roberta, so it still trains

## Wrap the 6 matrices of each of the 12 layers (setattr(obj, "query", x) assigns obj.query = x)
for layer in model.roberta.encoder.layer:
    for module, name in [(layer.attention.self, "query"), (layer.attention.self, "key"), (layer.attention.self, "value"),
                         (layer.attention.output, "dense"), (layer.intermediate, "dense"), (layer.output, "dense")]:
        setattr(module, name, LoRALinear(getattr(module, name), R, ALPHA, LORA_DROPOUT))
## The A's and B's were created after requires_grad_(False), so they're trainable

with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    s_after = model.roberta(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0]
print("B = 0, so adding LoRA leaves <s> unchanged:", torch.allclose(s_before, s_after))

trainable = [p for p in model.parameters() if p.requires_grad]   ## 72 A's, 72 B's and the head's 4 tensors
n_all, n_trainable = sum(p.numel() for p in model.parameters()), sum(p.numel() for p in trainable)
n_head = sum(p.numel() for p in model.head.parameters())
print(f"{sum(isinstance(m, LoRALinear) for m in model.modules())} LoRA layers. {n_all / 1e6:.1f}M parameters, "
      f"{n_trainable / 1e6:.2f}M of them trainable ({n_trainable / n_all * 100:.1f}%): "
      f"{(n_trainable - n_head) / 1e6:.2f}M in the A's and B's, {n_head / 1e6:.2f}M in the head")

## Optimizer and learning-rate schedule

The full fine-tune's AdamW and schedule (weight decay 0.1, β₂ 0.98, ε 1e-6, warmup over 6% of the steps, linear decay to 0, 10 epochs). The optimizer gets only `trainable`, and the learning rate is `LR`.

**Weight decay now pulls toward pre-trained RoBERTa.** It shrinks $A$ and $B$, and with them $\frac{\alpha}{r}BA$, the change to each $W$. In the full fine-tune it pulled every weight toward 0 instead.

In [ ]:
from transformers import get_linear_schedule_with_warmup

EPOCHS = 10
optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.1, betas=(0.9, 0.98), eps=1e-6)   ## only A, B and the head
total_steps = EPOCHS * len(train_loader)   ## 6250
warmup_steps = int(0.06 * total_steps)     ## 375
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()
print(f"learning rate {LR:.0e}, {total_steps} steps in total, {warmup_steps} of them warmup")

## Evaluation

As in the full fine-tune (dropout off, no gradients, bfloat16), but it also returns every post's 20 probabilities, which are saved at the end.

In [ ]:
## Returns (mean loss, accuracy, probabilities (number of posts, 20)) over all the batches of one DataLoader
def evaluate(loader):
    model.eval()
    logits, labels = [], []
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        for input_ids, attention_mask, y in loader:   ## (N, T), (N, T), (N,)
            logits.append(model(input_ids.to(device), attention_mask.to(device)).float().cpu())   ## (N, 20)
            labels.append(y)
    logits, labels = torch.cat(logits), torch.cat(labels)   ## (number of posts, 20), (number of posts,)
    probs = logits.softmax(dim=1)                           ## each row: one post's 20 probabilities, summing to 1
    return criterion(logits, labels).item(), (probs.argmax(dim=1) == labels).float().mean().item(), probs.numpy()

## Training loop

The full fine-tune's loop, with three changes:
- **Clipping covers only `trainable`,** the only weights with gradients.
- **The best epoch's copy holds only the trainable weights** (3.3M numbers instead of 125M): the frozen ones are the same in every epoch.
- **Each epoch's dev scores also go into `history`,** saved to `../sweeps/sweep.jsonl` at the end in `sweep.py`'s format, so that `summarize.py` lists the LoRA runs next to the full fine-tunes.

In [ ]:
import time

best = {"dev_acc": 0.0, "epoch": 0, "state": None}
history = []   ## one dict per epoch, in sweep.py's format
start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for step, (input_ids, attention_mask, labels) in enumerate(train_loader, start=1):
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(input_ids, attention_mask)   ## (N, 20)
        loss = criterion(logits.float(), labels)
        loss.backward()      ## gradients for the A's, B's and the head only: RoBERTa's own weights are frozen
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item() * len(labels)
        if step % 100 == 0:
            print(f"  epoch {epoch}, step {step}/{len(train_loader)}: loss {loss.item():.3f}, "
                  f"learning rate {scheduler.get_last_lr()[0]:.1e}")

    dev_loss, dev_acc, _ = evaluate(dev_loader)
    history.append({"epoch": epoch, "train_loss": round(train_loss / len(X_tr), 4), "dev_loss": round(dev_loss, 4),
                    "dev_acc": round(dev_acc * 100, 2), "seconds": round(time.perf_counter() - start)})
    print(f"epoch {epoch}: train loss {train_loss / len(X_tr):.3f}  dev loss {dev_loss:.3f}  dev acc {dev_acc*100:.2f}  "
          f"({time.perf_counter() - start:.0f} s so far)")
    if dev_acc >= best["dev_acc"]:   ## >=: on a tie, keep the later epoch
        best = {"dev_acc": dev_acc, "epoch": epoch,
                "state": {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}}

## strict=False: the copy has only the trainable weights, so the frozen ones are "missing" from it
model.load_state_dict(best["state"], strict=False)
print(f"Using epoch {best['epoch']} (highest dev accuracy). Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## Save the probabilities; test only in the final runs

The best epoch's dev and test probabilities go to `../sweeps/probs/`, named the way `sweep.py` names its files, and the run's dev history goes to `../sweeps/sweep.jsonl`. Then `summarize.py --final` can score all the seeds of the chosen learning rate, and their ensemble, at once. The test accuracy prints only with `SCORE_TEST = True`: while choosing the learning rate, don't look at test.

In [ ]:
import json
from pathlib import Path
from sklearn import metrics

_, _, dev_probs = evaluate(dev_loader)                    ## (1109, 20)
test_loss, test_acc, test_probs = evaluate(test_loader)   ## (7370, 20)
Path("../sweeps/probs").mkdir(exist_ok=True)
np.save(f"../sweeps/probs/{NAME}_seed{SEED}_acc_dev.npy", dev_probs)
np.save(f"../sweeps/probs/{NAME}_seed{SEED}_acc_test.npy", test_probs)
record = {"name": NAME,
          "cfg": {"lora_r": R, "lora_alpha": ALPHA, "lora_dropout": LORA_DROPOUT, "lr": LR, "epochs": EPOCHS,
                  "wd": 0.1, "warmup": 0.06, "beta2": 0.98, "eps": 1e-6, "seed": SEED, "select": "acc"},
          "history": history, "peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 1), "epoch_used": best["epoch"]}
with open("../sweeps/sweep.jsonl", "a") as f:   ## "a" appends a line, keeping the earlier runs
    f.write(json.dumps(record) + "\n")
print(f"Saved ../sweeps/probs/{NAME}_seed{SEED}_acc_{{dev,test}}.npy and added this run to ../sweeps/sweep.jsonl")

if SCORE_TEST:
    test_pred = test_probs.argmax(axis=1)
    print(f"\nLoRA-fine-tuned RoBERTa test accuracy: {test_acc*100:.2f}  "
          f"(macro-F1 {metrics.f1_score(y_test, test_pred, average='macro')*100:.2f})\n")
    rows = [("RoBERTa <s>, frozen + LR (tuned)", 67.23),
            ("RoBERTa <s>, frozen + MLP (tuned, mean of 10 seeds)", 68.33),
            ("RoBERTa, all weights fine-tuned (mean of all 4 runs)", 72.69),
            ("RoBERTa, all weights fine-tuned (mean of the 3 sweep.py seeds)", 73.13),
            (f"RoBERTa, LoRA (this run: {NAME}, seed {SEED})", test_acc * 100)]
    for name, acc in rows:
        print(f"{name:64s} {acc:6.2f}")
    print(f"(1 point = {len(y_test) / 100:.0f} test messages)")
else:
    print("Test not scored (SCORE_TEST = False)")

## Result

Run on 2026-11-02. Details are in `fine-tuning-results.md`, under "Learning-rate sweep" and conjecture 2.
- **Learning rate: 5e-4,** with the best dev accuracy: 78.18, against 77.55 (1e-4) and 77.73 (2e-4). 5e-4 was the top of the grid, so 1e-3 was added before choosing. It was unstable in the first epochs and ended lower (77.37).
- **Conjecture 2: a tie.** Test 73.23, 73.34 and 73.31 (seeds 423, 1 and 2): mean 73.29, against the full fine-tune's 73.13 (+0.16).
- **The ensembles: LoRA's is better,** 75.07 vs. 73.93 (321 vs. 237 posts right only in one, McNemar p = 0.0004). Each LoRA run is about as good as a fully fine-tuned one, but the runs disagree more (on 16.9% of test posts per pair, vs. 14.0%), so averaging fixes more errors.
- **Were 10 epochs enough?** Probably. Dev accuracy leveled off by epoch 7 in all 3 seeds, and each run's last three epochs were within about 0.8 of its best. But the learning rate decays to 0 by epoch 10, which flattens the end of any run, so a test of more epochs needs a longer schedule.
- **Cost:** 5.1 GB peak and about 48 s per epoch, against the full fine-tune's 5.2 GB and 46 s. No saving at this size: with 16 posts × 512 tokens, the activations take most of the memory, not the gradients and AdamW averages that LoRA saves.
- **Rank barely matters:** r = 4 and r = 64 (α still 32) scored 72.75 and 73.41 (3-seed means), within 1 point of r = 16's 73.29.